In [1]:
import pandas as pd
import re

TRAIN_FILE = "train_br_augmented_safe.xlsx"
VALID_FILE = "valid.xlsx"
TEST_FILE  = "test.xlsx"

TEXT_COL = "text_ar"
GLOSS_COL = "gloss_ar"

train_df = pd.read_excel(TRAIN_FILE)
valid_df = pd.read_excel(VALID_FILE)
test_df  = pd.read_excel(TEST_FILE)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape :", test_df.shape)

Train shape: (122, 22)
Valid shape: (13, 7)
Test shape : (13, 7)


In [2]:
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [3]:
for df_name, df_ in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in [TEXT_COL, GLOSS_COL] if c not in df_.columns]
    if missing:
        raise ValueError(f"{df_name} is missing columns: {missing}")

    df_[TEXT_COL] = df_[TEXT_COL].astype(str).apply(normalize_ar)
    df_[GLOSS_COL] = df_[GLOSS_COL].astype(str).apply(normalize_ar)

print("Columns checked and normalized.")

Columns checked and normalized.


In [4]:
def clean_pair_df(df_):
    df_ = df_.dropna(subset=[TEXT_COL, GLOSS_COL]).copy()
    df_ = df_[(df_[TEXT_COL] != "") & (df_[GLOSS_COL] != "")].copy()
    df_ = df_.drop_duplicates(subset=[TEXT_COL, GLOSS_COL]).reset_index(drop=True)
    return df_

train_df = clean_pair_df(train_df)
valid_df = clean_pair_df(valid_df)
test_df  = clean_pair_df(test_df)

print("Train cleaned:", train_df.shape)
print("Valid cleaned:", valid_df.shape)
print("Test cleaned :", test_df.shape)

Train cleaned: (122, 22)
Valid cleaned: (13, 7)
Test cleaned : (13, 7)


In [5]:
TASK_PREFIX = "translate Arabic to gloss: "

def build_t5_df(df_):
    out = pd.DataFrame()
    out["input_text"] = TASK_PREFIX + df_[TEXT_COL].astype(str)
    out["target_text"] = df_[GLOSS_COL].astype(str)
    return out

train_t5 = build_t5_df(train_df)
valid_t5 = build_t5_df(valid_df)
test_t5  = build_t5_df(test_df)

train_t5.head(10)

,input_text,target_text
0,translate Arabic to gloss: انتبه,انتبه خطر تمام
1,translate Arabic to gloss: حبة 3 مرة قبل الاكل...,3 مرة اكل قبل نص ساعة حبة
2,translate Arabic to gloss: حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة
3,translate Arabic to gloss: حبة 3 مرة قبل الاكل...,3 مرة اكل قبل ساعة حبة
4,translate Arabic to gloss: 6 شهر,6 شهر
5,translate Arabic to gloss: نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة
6,translate Arabic to gloss: لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام
7,translate Arabic to gloss: خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا
8,translate Arabic to gloss: بعد الاكل,اكل خلص فورا دواء تمام
9,translate Arabic to gloss: ممكن يعملك شوية اسه...,انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام


In [6]:
print("Train T5:", train_t5.shape)
print("Valid T5:", valid_t5.shape)
print("Test T5 :", test_t5.shape)

print("\nSample train rows:")
display(train_t5.head(10))

Train T5: (122, 2)
Valid T5: (13, 2)
Test T5 : (13, 2)

Sample train rows:


,input_text,target_text
0,translate Arabic to gloss: انتبه,انتبه خطر تمام
1,translate Arabic to gloss: حبة 3 مرة قبل الاكل...,3 مرة اكل قبل نص ساعة حبة
2,translate Arabic to gloss: حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة
3,translate Arabic to gloss: حبة 3 مرة قبل الاكل...,3 مرة اكل قبل ساعة حبة
4,translate Arabic to gloss: 6 شهر,6 شهر
5,translate Arabic to gloss: نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة
6,translate Arabic to gloss: لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام
7,translate Arabic to gloss: خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا
8,translate Arabic to gloss: بعد الاكل,اكل خلص فورا دواء تمام
9,translate Arabic to gloss: ممكن يعملك شوية اسه...,انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام


In [7]:
train_t5.to_csv("train_t5.csv", index=False, encoding="utf-8-sig")
valid_t5.to_csv("valid_t5.csv", index=False, encoding="utf-8-sig")
test_t5.to_csv("test_t5.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- train_t5.csv")
print("- valid_t5.csv")
print("- test_t5.csv")

Saved:
- train_t5.csv
- valid_t5.csv
- test_t5.csv


In [8]:
train_t5.to_excel("train_t5.xlsx", index=False)
valid_t5.to_excel("valid_t5.xlsx", index=False)
test_t5.to_excel("test_t5.xlsx", index=False)

print("Saved optional Excel files.")

Saved optional Excel files.


In [9]:
if "source_type" in train_df.columns:
    print(train_df["source_type"].value_counts(dropna=False))
else:
    print("source_type column not found.")

source_type
original     102
augmented     20
Name: count, dtype: int64


In [10]:
if "source_type" in train_df.columns:
    aug_only = train_df[train_df["source_type"] == "augmented"].copy()
    print("Augmented rows:", len(aug_only))
    display(aug_only[[TEXT_COL, GLOSS_COL]].head(20))
else:
    print("source_type column not found.")

Augmented rows: 20


,text_ar,gloss_ar
102,حبة 3 مرة قبل الطعام بنص ساعة .,3 مرة اكل قبل نص ساعة حبة
103,حبة 3 مرة قبل وجبة بنص ساعة .,3 مرة اكل قبل نص ساعة حبة
104,حبة 3 مرة قبل الطعام .,3 مرة اكل قبل حبة
105,حبة 3 مرة قبل وجبة .,3 مرة اكل قبل حبة
106,حبة 3 مرة قبل الطعام بساعة .,3 مرة اكل قبل ساعة حبة
107,حبة 3 مرة قبل وجبة بساعة .,3 مرة اكل قبل ساعة حبة
108,خده ورا الطعام,اكل بعد حبة تمام معدة فاضية لا
109,خده ورا وجبة,اكل بعد حبة تمام معدة فاضية لا
110,بعد الطعام,اكل خلص فورا دواء تمام
111,بعد وجبة,اكل خلص فورا دواء تمام
